# Bank Marketing: Individual Fairness Preprocessing

Author: Ilse Harmers \
Last modified: March 9, 2026

Certain sections of code in this notebook have been kindly provided by dr. Mina Alishahi.

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from snsynth import Synthesizer
import snsynth.transform as tf
import utils

from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

In [ ]:
# Initializing our random forest classifier.
rf_clf = RandomForestClassifier()

def encode_ord(sample):
    """This function encodes a Bank-based synthetic dataset such that all ordinal columns are scaled to a range of (0, 1)."""

    sample["day"] = ((sample["day"] - sample["day"].min()) / (sample["day"].max() - sample["day"].min()))
    
    return sample

def compute_similarity_fairness(indices, predictions, distances, neighbors):
    """
    Compute individual fairness for a chunk of data using precomputed k-NN distances and neighbors.

    Parameters:
        indices (list): Indices of the chunk.
        predictions (array-like): Model predictions for individuals.
        distances (array-like): Distances from k-NN.
        neighbors (array-like): Indices of k-NN neighbors.

    Returns:
        float: Mean individual fairness score for the chunk.

    Code provided by: dr. Mina Alishahi
    """
    
    fairness_scores = []
    for idx in indices:
        # Iterate through the k-nearest neighbors (skip the first as it's the individual itself).
        for neighbor_idx, distance in zip(neighbors[idx][1:], distances[idx][1:]):
            if distance < 1e-6:  # Treat very small distances as non-zero.
                distance = 1e-6
            pred_diff = abs(predictions[idx] - predictions[neighbor_idx])
            fairness_scores.append(pred_diff * distance)
    return np.mean(fairness_scores) if fairness_scores else 0.0

def similarity_fairness(predictions, features, similarity_metric = "euclidean", k = 100, num_chunks = 8):
    """
    Compute individual fairness for model predictions using k-NN.

    Parameters:
        predictions (array-like): Model predictions for individuals.
        features (array-like): Feature matrix (N x D).
        similarity_metric (str): Similarity metric ('cosine' or 'euclidean').
        k (int): Number of nearest neighbors to consider.
        num_chunks (int): Number of chunks.

    Returns:
        float: Average individual fairness score (lower is better).

    Code provided by: dr. Mina Alishahi
    """
    
    n = len(features)
    if k >= n:
        raise ValueError(f"Invalid k: {k}. Must be less than the number of data points: {n}.")
    
    # Fit k-NN on the features.
    np.random.seed(42)   # setting random seed
    knn = NearestNeighbors(n_neighbors = k + 1, metric = similarity_metric).fit(features)
    distances, neighbors = knn.kneighbors(features, return_distance=True)

    # Divide indices into chunks.
    chunk_size = max(1, n // num_chunks)
    fairness_scores = [
        compute_similarity_fairness(
            range(start, min(start + chunk_size, n)), predictions, distances, neighbors
        )
        for start in range(0, n, chunk_size)
    ]

    # Gather results from all chunks and compute the overall mean.
    return np.mean(fairness_scores) if fairness_scores else 0.0


def compute_neighborhood_consistency_fairness(indices, predictions, distances, neighbors):
    """
    Compute neighborhood consistency for a chunk of data.

    Parameters:
        indices (list): Indices of the chunk.
        predictions (array-like): Model predictions for individuals.
        distances (array-like): Distances from k-NN.
        neighbors (array-like): Indices of k-NN neighbors.

    Returns:
        float: Mean neighborhood consistency score for the chunk.

    Code provided by: dr. Mina Alishahi
    """
    consistency_scores = []
    for idx in indices:
        # Calculate consistency score for the current individual.
        local_consistency = np.mean([
            abs(predictions[idx] - predictions[neighbor_idx])
            for neighbor_idx in neighbors[idx][1:]  # Skip the first neighbor (the individual itself).
        ])
        consistency_scores.append(local_consistency)

    return np.mean(consistency_scores)

def neighborhood_consistency_fairness(predictions, features, similarity_metric = "euclidean", k = 100, num_chunks = 8):
    """
    Compute neighborhood consistency metric using k-NN.

    Parameters:
        predictions (array-like): Model predictions for individuals.
        features (array-like): Feature matrix (N x D).
        similarity_metric (str): Similarity metric ('cosine' or 'euclidean').
        k (int): Number of nearest neighbors to consider.
        num_chunks (int): Number of chunks.

    Returns:
        float: Average neighborhood consistency score (lower is better).

    Code provided by: dr. Mina Alishahi
    """
        
    # Fit k-NN on the features.
    np.random.seed(42)   # setting random seed
    knn = NearestNeighbors(n_neighbors = k + 1, metric = similarity_metric).fit(features)
    distances, neighbors = knn.kneighbors(features, return_distance = True)

    # Divide indices into chunks.
    n = len(features)
    chunk_size = n // num_chunks
    consistency_scores = [
        compute_neighborhood_consistency_fairness(
            range(start, min(start + chunk_size, n)), predictions, distances, neighbors
        )
        for start in range(0, n, chunk_size)
    ]
    
    # Gather results from all chunks and compute the overall mean.
    return np.mean(consistency_scores)

def fairness_analysis(train_set, test_set, y):
    """This function computes the similarity fairness and neigborhood consistency fairness scores for a given train and test set.
    
    train_set [array-like]: train set 
    test_set [array-like]: test set
    y [string]: column name of target attribute
    """

    np.random.seed(42)   # setting random seed
    rf_model = rf_clf.fit(train_set.drop(columns = [y]), train_set[y])
    soft_ypred = rf_model.predict_proba(test_set.drop(columns = [y]))
    ypred = rf_model.predict(test_set.drop(columns = [y]))

    # Min-max scaling the ordinal columns.
    test_copy = test_set.copy()
    test_enc = encode_ord(test_copy)
        
    sf_score = similarity_fairness(soft_ypred, test_enc.drop(columns = [y]).values)

    ncf_score = neighborhood_consistency_fairness(ypred, test_enc.drop(columns = [y]).values)

    return (sf_score, ncf_score)

def bank_encode(df):
    """This function encodes a Bank-based synthetic dataset such that all categorical columns are (one-hot) encoded and all numerical columns are
    scaled to a range of (0, 1).
    
    df [pandas DataFrame]: synthetic dataset
    """
    
    # One-hot encoding a Bank-based dataset. 
    cat_columns = df.select_dtypes(include=['object']).columns.to_list()
    train_cat_encoded = utils.one_hot_encode(df[cat_columns], order = [["no", "yes"], ["no", "yes"], ["no", "yes"],
                                             ["no", "yes"]])

    # Transforming the continuous columns with the TableTransformer function.
    tt = tf.TableTransformer([
        tf.MinMaxTransformer(lower = df["age"].min(), upper = df["age"].max(),
                             negative = False), # age; scaling to range (0, 1)
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = -1 * (np.log(abs(df["balance"].min()) + 1)),
                                 upper = np.log(df["balance"].max() + 1), 
                                 negative = False) # balance; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = 0, upper = np.log(df["duration"].max() + 1), 
                                 negative = False) # duration; scaling to range (0, 1)
        ]),
        tf.MinMaxTransformer(lower = df["campaign"].min(), upper = df["campaign"].max(),
                             negative = False), # campaign; scaling to range (0, 1)
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = -1 * (np.log(abs(df["pdays"].min()) + 1)), 
                                 upper = np.log(df["pdays"].max() + 1),
                                 negative = False) # pdays; scaling to range (0, 1)
        ]),
        tf.ChainTransformer([
            tf.LogModulusTransformer(),
            tf.MinMaxTransformer(lower = 0, upper = np.log(df["previous"].max() + 1),
                                 negative = False) # previous; scaling to range (0, 1)
        ]),
    ])

    tf_cont_cols = np.array(synth._get_train_data(
                        df[["age", "balance", "duration", "campaign", "pdays", "previous"]], transformer = tt, 
                        preprocessor_eps = 0.0, style = 'gan', nullable = False, categorical_columns=None, 
                        ordinal_columns=None, continuous_columns=None,))

    num_encoded = pd.DataFrame(tf_cont_cols, columns = ["age", "balance", "duration", "campaign", "pdays", "previous"])

    # Concatenating all the columns back together.
    train_encoded = pd.concat([num_encoded.reset_index(drop = True), train_cat_encoded.reset_index(drop = True), 
                               df[["day"]].reset_index(drop = True)], axis = 1)
    
    return train_encoded

## Pre-Processing of Bank Marketing Dataset

In [ ]:
# Importing Bank Marketing dataset (downloaded from https://archive.ics.uci.edu/dataset/222/bank+marketing on June 1, 2025).

# Assigning file path for importing the dataset. 
path = "../bank-marketing/bank-full.csv"

# Importing dataset as pandas DataFrame.
bank = pd.read_csv(path, sep = ";")

## Individual Fairness Preprocessing

In [ ]:
# Initializing the synthesizer for preprocessing the Bank dataset with the TableTransformer function. 
synth = Synthesizer.create('pategan', epsilon = 10.0, delta = 1e-05, plot_losses = True)

bank_train, bank_test = train_test_split(bank, train_size = 0.80, random_state = 42)
train_encoded = bank_encode(bank_train)
test_encoded = bank_encode(bank_test)

In [ ]:
test_encoded.describe()#, train_encoded.describe()

### "No Pre-Processing" Reference

In [ ]:
# Individual fairness metrics.
bank_sf, bank_ncf = fairness_analysis(train_set = train_encoded, test_set = test_encoded, y = "y")
print("sf_score: ", bank_sf)  
print("ncf_score: ", bank_ncf)

# Encoding the target and sensitive attributes for the group fairness metrics.
y_encoded = utils.one_hot_encode(bank_train[["y"]], order = [["no", "yes"]])
age_encoded = bank_train["age"].copy()
age_encoded.loc[age_encoded <= 25] = 1
age_encoded.loc[age_encoded > 25] = 0
cols_encoded = pd.concat([age_encoded.reset_index(drop = True), y_encoded.reset_index(drop = True)], axis = 1)
print("Demographic parity:", utils.demographic_parity(df = cols_encoded, s = "age", y = "y"))
print("Disparate impact:", utils.disparate_impact(df = cols_encoded, s = "age", y = "y"))

np.random.seed(42)
rf = rf_clf.fit(train_encoded.drop(columns = ["y"]), train_encoded["y"])
ypred = rf.predict(test_encoded.drop(columns = ["y"]))
auroc = roc_auc_score(test_encoded["y"], ypred)
print(f"AUROC on test data: {auroc:.8f}")

## Train KMeans Clustering Algorithm

In [ ]:
# Min-max scaling the ordinal columns.
train_ord = train_encoded.copy()
train_nom = encode_ord(train_ord)
test_ord = test_encoded.copy()
test_nom = encode_ord(test_ord)

In [ ]:
clusters = np.arange(50, 600, 50)
clusters_test = np.arange(50, 600, 50)
thres = 1.0
for clus in range(len(clusters)):
    # Training the KMeans clustering algorithm on the min-max scaled train set.
    print(f"Cluster size (train) = {clusters[clus]}")
    kmeans = KMeans(n_clusters = clusters[clus], random_state = 0, n_init = "auto").fit(train_nom.drop(columns = ["y"]))
    pred = kmeans.predict(train_nom.drop(columns = ["y"]))
    print("kmeans inertia:", kmeans.inertia_)

    train_kmeans = train_nom.copy()
    train_kmeans["cluster"] = pred

    # If an entry in the train set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
    # re-assigned to the majority label of the cluster (if this is not the case already).
    distances = [pairwise_distances(
                    np.array([kmeans.cluster_centers_[c]]), train_kmeans[train_kmeans["cluster"] == c].drop(columns = ["y", "cluster"]).values
                    )[0] for c in np.unique(pred)]
    for c in range(len(np.unique(pred))):
        index = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]].iloc[np.where(distances[c] <= thres)].index
        train_kmeans.loc[index, "y"] = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]]["y"].value_counts().index[0]

    print("KMeans (y = 1):", train_kmeans[train_kmeans["y"] == 1].shape)
    print("Original train set (y = 1):", train_encoded[train_encoded["y"] == 1].shape)

    # Training the KMeans clustering algorithm on the min-max scaled test set.
    print(f"Cluster size (test) = {clusters_test[clus]}")
    kmeans = KMeans(n_clusters = clusters_test[clus], random_state = 3, n_init = "auto").fit(test_nom.drop(columns = ["y"]))
    pred_test = kmeans.predict(test_nom.drop(columns = ["y"]))

    test_kmeans = test_nom.copy()
    test_kmeans["cluster"] = pred_test

    # If an entry in the test set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
    # re-assigned to the majority label of the cluster (if this is not the case already).
    distances = [pairwise_distances(
                    np.array([kmeans.cluster_centers_[c]]), test_kmeans[test_kmeans["cluster"] == c].drop(columns = ["y", "cluster"]).values
                    )[0] for c in np.unique(pred_test)]
    for c in range(len(np.unique(pred_test))):
        index = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]].iloc[np.where(distances[c] <= thres)].index
        test_kmeans.loc[index, "y"] = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]]["y"].value_counts().index[0]

    print("KMeans (y = 1):", test_kmeans[test_kmeans["y"] == 1].shape)
    print("Original test set (y = 1):", test_encoded[test_encoded["y"] == 1].shape)

    ctrain = train_encoded.copy()
    ctrain["y"] = train_kmeans["y"]
    ctest = test_encoded.copy()
    ctest["y"] = test_kmeans["y"]
    
    bank_sf, bank_ncf = fairness_analysis(train_set = ctrain, test_set = ctest, y = "y")
    print("sf_score: ", bank_sf)  
    print("ncf_score: ", bank_ncf)

    np.random.seed(42)
    rf = rf_clf.fit(ctrain.drop(columns = ["y"]), ctrain["y"])
    ypred = rf.predict(ctest.drop(columns = ["y"]))
    auroc = roc_auc_score(ctest["y"], ypred)
    print(f"AUROC on test data: {auroc:.8f}")
    print()

In [ ]:
clusters = [250] * 11
clusters_test = np.arange(50, 600, 50)
thres = 1.0 
for clus in range(len(clusters)):
    # Training the KMeans clustering algorithm on the min-max scaled train set.
    print(f"Cluster size (train) = {clusters[clus]}")
    kmeans = KMeans(n_clusters = clusters[clus], random_state = 0, n_init = "auto").fit(train_nom.drop(columns = ["y"]))
    pred = kmeans.predict(train_nom.drop(columns = ["y"]))
    print("kmeans inertia:", kmeans.inertia_)

    train_kmeans = train_nom.copy()
    train_kmeans["cluster"] = pred

    # If an entry in the train set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
    # re-assigned to the majority label of the cluster (if this is not the case already).
    distances = [pairwise_distances(
                    np.array([kmeans.cluster_centers_[c]]), train_kmeans[train_kmeans["cluster"] == c].drop(columns = ["y", "cluster"]).values
                    )[0] for c in np.unique(pred)]
    for c in range(len(np.unique(pred))):
        index = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]].iloc[np.where(distances[c] <= thres)].index
        train_kmeans.loc[index, "y"] = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]]["y"].value_counts().index[0]

    print("KMeans (y = 1):", train_kmeans[train_kmeans["y"] == 1].shape)
    print("Original train set (y = 1):", train_encoded[train_encoded["y"] == 1].shape)

    # Training the KMeans clustering algorithm on the min-max scaled test set.
    print(f"Cluster size (test) = {clusters_test[clus]}")
    kmeans = KMeans(n_clusters = clusters_test[clus], random_state = 3, n_init = "auto").fit(test_nom.drop(columns = ["y"]))
    pred_test = kmeans.predict(test_nom.drop(columns = ["y"]))

    test_kmeans = test_nom.copy()
    test_kmeans["cluster"] = pred_test

    # If an entry in the test set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
    # re-assigned to the majority label of the cluster (if this is not the case already).
    distances = [pairwise_distances(
                    np.array([kmeans.cluster_centers_[c]]), test_kmeans[test_kmeans["cluster"] == c].drop(columns = ["y", "cluster"]).values
                    )[0] for c in np.unique(pred_test)]
    for c in range(len(np.unique(pred_test))):
        index = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]].iloc[np.where(distances[c] <= thres)].index
        test_kmeans.loc[index, "y"] = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]]["y"].value_counts().index[0]

    print("KMeans (y = 1):", test_kmeans[test_kmeans["y"] == 1].shape)
    print("Original test set (y = 1):", test_encoded[test_encoded["y"] == 1].shape)

    ctrain = train_encoded.copy()
    ctrain["y"] = train_kmeans["y"]
    ctest = test_encoded.copy()
    ctest["y"] = test_kmeans["y"]
    
    bank_sf, bank_ncf = fairness_analysis(train_set = ctrain, test_set = ctest, y = "y")
    print("sf_score: ", bank_sf)  
    print("ncf_score: ", bank_ncf)

    np.random.seed(42)
    rf = rf_clf.fit(ctrain.drop(columns = ["y"]), ctrain["y"])
    ypred = rf.predict(ctest.drop(columns = ["y"]))
    auroc = roc_auc_score(ctest["y"], ypred)
    print(f"AUROC on test data: {auroc:.8f}")
    print()

In [ ]:
clus_train = 250
clus_test = 200
thres = 1.0

# Training the KMeans clustering algorithm on the min-max scaled train set.
print(f"Cluster size (train) = {clus_train}")
kmeans = KMeans(n_clusters = clus_train, random_state = 0, n_init = "auto").fit(train_nom.drop(columns = ["y"]))
pred = kmeans.predict(train_nom.drop(columns = ["y"]))
print("kmeans inertia:", kmeans.inertia_)

train_kmeans = train_nom.copy()
train_kmeans["cluster"] = pred

# If an entry in the train set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
# re-assigned to the majority label of the cluster (if this is not the case already).
distances = [pairwise_distances(
                np.array([kmeans.cluster_centers_[c]]), train_kmeans[train_kmeans["cluster"] == c].drop(columns = ["y", "cluster"]).values
                )[0] for c in np.unique(pred)]
for c in range(len(np.unique(pred))):
    index = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]].iloc[np.where(distances[c] <= thres)].index
    train_kmeans.loc[index, "y"] = train_kmeans[train_kmeans["cluster"] == np.unique(pred)[c]]["y"].value_counts().index[0]

print("Train set (y = yes):", bank_train[bank_train["y"] == "yes"].shape)
print("KMeans (y = 1):", train_kmeans[train_kmeans["y"] == 1].shape)
new_train = bank_train.copy()
new_train["y"] = train_kmeans["y"].values
print("New train set (y = 1):", new_train[new_train["y"] == 1].shape)
new_train["y"] = new_train["y"].replace({0: "no", 1: "yes"})
print("New train set (y = yes):", new_train[new_train["y"] == "yes"].shape)
new_train.to_csv("train-test-datasets/bank-marketing/bank_train.csv", index = False)
print()

# Training the KMeans clustering algorithm on the min-max scaled test set.
print(f"Cluster size (test) = {clus_test}")
kmeans = KMeans(n_clusters = clus_test, random_state = 3, n_init = "auto").fit(test_nom.drop(columns = ["y"]))
pred_test = kmeans.predict(test_nom.drop(columns = ["y"]))

test_kmeans = test_nom.copy()
test_kmeans["cluster"] = pred_test

# If an entry in the test set has a distance to its assigned cluster's center smaller than the threshold, the entry's target label is 
# re-assigned to the majority label of the cluster (if this is not the case already).
distances = [pairwise_distances(
                np.array([kmeans.cluster_centers_[c]]), test_kmeans[test_kmeans["cluster"] == c].drop(columns = ["y", "cluster"]).values
                )[0] for c in np.unique(pred_test)]
for c in range(len(np.unique(pred_test))):
    index = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]].iloc[np.where(distances[c] <= thres)].index
    test_kmeans.loc[index, "y"] = test_kmeans[test_kmeans["cluster"] == np.unique(pred_test)[c]]["y"].value_counts().index[0]

print("Test set (y = yes):", bank_test[bank_test["y"] == "yes"].shape)
print("KMeans (y = 1):", test_kmeans[test_kmeans["y"] == 1].shape)
new_test = bank_test.copy()
new_test["y"] = test_kmeans["y"].values
print("New test set (y = 1):", new_test[new_test["y"] == 1].shape)
new_test["y"] = new_test["y"].replace({0: "no", 1: "yes"})
print("New test set (y = yes):", new_test[new_test["y"] == "yes"].shape)
new_test.to_csv("train-test-datasets/bank-marketing/bank_test.csv", index = False)
print()

ntrain_encoded = bank_encode(new_train)
ntest_encoded = bank_encode(new_test)
# Checking for missing columns between the train and test set. 
missing_cols = list(set(list(ntrain_encoded.columns)) - set(list(ntest_encoded.columns)))
print("Missing column(s):", missing_cols)
# If there is a missing column, most likely due to a label from the train set not being present in the test set, 
# then we reinsert a 'zero' column into the test set at the right place to account for this missing label.
if missing_cols:
    for c in missing_cols:
        df_col = pd.DataFrame({c: ntest_encoded["age"]*0})
        ntest_encoded.insert(0, c, value = df_col)
ntest_encoded = ntest_encoded[ntrain_encoded.columns]

bank_sf, bank_ncf = fairness_analysis(train_set = ntrain_encoded, test_set = ntest_encoded, y = "y")
print("sf_score: ", bank_sf)  
print("ncf_score: ", bank_ncf)

np.random.seed(42)
rf = rf_clf.fit(ntrain_encoded.drop(columns = ["y"]), ntrain_encoded["y"])
ypred = rf.predict(ntest_encoded.drop(columns = ["y"]))
auroc = roc_auc_score(ntest_encoded["y"], ypred)
print(f"AUROC on test data: {auroc:.8f}")